In [23]:
import logic.pulsed.pulse_objects as po
from logic.pulsed.sampling_functions import SamplingFunctions as SF
import time
import matplotlib.pyplot as plt
pjl = pulsedjupyterlogic_AWG
import pickle
import datetime
import numpy as np

In [24]:
def measure_PODMR():
    pulsedmeasurementlogic_AWG._PulsedMeasurementLogic__elapsed_sweeps = 0
    pjl.initialize_ensemble(laser_power_voltage = laser_power_voltage, pi_pulse=pi_pulse, pi_pulse_1=pi_pulse_1, LO_freq_0=LO_freq_0, target_freq_0=target_freq_0, power_0=power_0, LO_freq_1=LO_freq_1, target_freq_1=target_freq_1, power_1=power_1, switch_MW = switch_MW)
    pjl.PODMR(mw_start, mw_stop, mw_step)
    pjl.start_measurement(measurement_type='PODMR', tip_name=tip_name, sample=sample, 
                          temperature=temperature, b_field=b_field, contact=contact, extra=extra)
    
    while True and not afm_scanner_logic.jupyter_meas_stop:
        if pulsedmasterlogic_AWG.elapsed_sweeps>max_sweeps_PODMR:
            pulsedmeasurementlogic_AWG.stop_pulsed_measurement()
            time.sleep(0.1)
            pulsedmeasurement_AWG._pa.fit_param_fit_func_ComboBox.setCurrentFit(PODMR_fit)
            current_fit_method = pulsedmeasurement_AWG._pa.fit_param_fit_func_ComboBox.getCurrentFit()[0]
            use_alt_data = False
            time.sleep(0.1)
            pulsedmeasurement_AWG.pulsedmasterlogic().do_fit(current_fit_method, use_alt_data)
            time.sleep(0.1)
            pulsedmeasurement_AWG.save_clicked()
            return pulsedmeasurementlogic_AWG.fit_result.params['center'].value
        else:
            time.sleep(0.01)
            
def measure_T1():
    pulsedmeasurementlogic_AWG._PulsedMeasurementLogic__elapsed_sweeps = 0
    pjl.initialize_ensemble(laser_power_voltage = laser_power_voltage, pi_pulse=pi_pulse, pi_pulse_1=pi_pulse_1, LO_freq_0=LO_freq_0, target_freq_0=target_freq_0, power_0=power_0, LO_freq_1=LO_freq_1, target_freq_1=target_freq_1, power_1=power_1, switch_MW = switch_MW)
    pjl.T1_dark_init_alt_exp(tau_start, tau_stop, tau_num)
    pjl.start_measurement(measurement_type='T1-dark-init-alt', tip_name=tip_name, sample=sample, 
                          temperature=temperature, b_field=b_field, contact=contact, extra=extra)

    while True and not afm_scanner_logic.jupyter_meas_stop:
        if pulsedmasterlogic_AWG.elapsed_sweeps>max_sweeps_T1:
            pulsedmeasurementlogic_AWG.stop_pulsed_measurement()
            pulsedmeasurementlogic_AWG.alternative_data_type = 'Delta'
            time.sleep(0.1)
            pulsedmeasurement_AWG._pa.fit_param_alt_fit_func_ComboBox.setCurrentFit(T1_fit)
            current_fit_method = pulsedmeasurement_AWG._pa.fit_param_alt_fit_func_ComboBox.getCurrentFit()[0]
            use_alt_data = True
            time.sleep(0.1)
            pulsedmeasurement_AWG.pulsedmasterlogic().do_fit(current_fit_method, use_alt_data)
            time.sleep(0.1)
            pulsedmeasurement_AWG.save_clicked()
            break
        else:
            time.sleep(0.01)
            
def goto(x_pos, y_pos):
    afm_scanner_logic.set_afm_pos({'x':x_pos, 'y':y_pos})
    

In [25]:
#measurement parameters
laser_power_voltage = podmrlogic.laser_power_voltage

#Driving information for the first LO (SMBV)
target_freq_0 = 2.9257e9
LO_freq_0 = target_freq_0 + 100e6
power_0 = -5
pi_pulse = 40e-9 

#For same NV rabi SGS is run with power_1 = power_0 - 11dBm
#Driving information for the second LO (SGS)
target_freq_1 = 2.87e9
LO_freq_1 = target_freq_1 + 100e6
power_1 = -100
pi_pulse_1 = 72e-9

switch_MW = False #If True, MW1 is used as the main MW source

#additional information for save tag
tip_name = 'A-H12-13'
sample = 'YBCO'
temperature = '10K'
b_field = '5mT_OOP_FC'
contact = 'IC'
extra = 'meissner'

afm_scanner_logic.jupyter_meas_stop = False

#### Run a T1 measurement with a defined time

In [4]:
meas_time_hour = 1.5
meas_time = meas_time_hour*60*60

tau_start = 1e-6
tau_stop = 100e-3
tau_num = 20

T1_fit = 'exp_decay'

afm_scanner_logic.jupyter_meas_stop = False

measure_T1()

#### Run a T1 measurement with a defined time at several points with PODMR resonance tracking

In [26]:
# meas_time_hour = 1.5
# meas_time = meas_time_hour*60*60

max_sweeps_T1 = 60000

tau_start = 1e-6
tau_stop = 50e-3
tau_num = 20

max_sweeps_PODMR = 150000

mw_start = 2.88e9
mw_stop = 3.0e9
mw_step = 1e6

x_pos_1 = 9.45e-6
y_pos_1 = 3.5e-6
extra_pos_1 = 'meissner_region'

x_pos_2 = 10.5e-6
y_pos_2 = 3.5e-6
extra_pos_2 = 'vortex_region'

PODMR_fit = 'gaussian'
T1_fit = 'exp_decay'

#measure pos 1
goto(x_pos_1,y_pos_1)
extra = extra_pos_1
target_freq_0 = measure_PODMR()
LO_freq_0 = target_freq_0 + 100e6
measure_T1()

#measure pos 2
goto(x_pos_2,y_pos_2)
extra = extra_pos_2
target_freq_0 = measure_PODMR()
LO_freq_0 = target_freq_0 + 100e6
measure_T1()

#measure OOC
spm.retract_probe()
extra = ''
contact = 'OOC'
target_freq_0 = measure_PODMR()
LO_freq_0 = target_freq_0 + 100e6
measure_T1()
